##S11 - Setting up the pipeline to collect information on contract data from three different source:

a. Rookie Scale from RealGM at season x draft pick grain
b. Minimum salary scale from RealGM at season x years of service grain
c. Contract details for non rookie deals and non minimum contract i.e all other market contracts for NBA players

### Salary Minimum Scale

In [1]:
import pandas as pd
import os
import time

salary_min_18 = pd.read_csv("../data/raw/contracts/Salary Minimum Scale 2018-2026.csv")

salary_min_01 = pd.read_csv("../data/raw/contracts/Salary Minimum Scale 2001-2017.csv")

print(salary_min_18.shape)
print(salary_min_01.shape)

(107, 9)
(35, 9)


In [2]:
print(salary_min_18.info())
print(salary_min_01.info())

<class 'pandas.DataFrame'>
RangeIndex: 107 entries, 0 to 106
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   YOS         107 non-null    str  
 1   Year 1      107 non-null    str  
 2   Year 2      98 non-null     str  
 3   Year 3      89 non-null     str  
 4   Year 4      80 non-null     str  
 5   Year 5      70 non-null     str  
 6   Season      107 non-null    str  
 7   Year        107 non-null    str  
 8   CBA Regime  105 non-null    str  
dtypes: str(9)
memory usage: 7.7 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   YOS           35 non-null     str  
 1   2011-2012     35 non-null     str  
 2   2012-2013     35 non-null     str  
 3   2013-2014     35 non-null     str  
 4   2014-2015     35 non-null     str  
 5   2015-2016     35 non-null     str  
 6   2016-2017    

In [9]:
salary_min_01.tail()


,YOS,2011-2012,2012-2013,2013-2014,2014-2015,2015-2016,2016-2017,CBA Regime,CBA Regime.1
30,6,"$500,000","$635,000","$673,500","$715,850","$762,435","$813,679","$870,046",1998
31,7,"$500,000","$697,500","$736,000","$778,350","$824,935","$876,179","$932,546",1998
32,8,"$500,000","$760,000","$798,500","$840,850","$887,435","$938,679","$995,046",1998
33,9,"$500,000","$885,000","$923,500","$965,850","$1,000,000","$1,000,000","$1,000,000",1998
34,10+,"$500,000","$1,000,000","$1,000,000","$1,000,000","$1,030,000","$1,070,000","$1,100,000",1998


In [10]:
salary_min_18.head(25)

,YOS,Year 1,Year 2,Year 3,Year 4,Year 5,Season,Year,CBA Regime
0,0 YOS,"$1,272,870",NaN,NaN,NaN,NaN,2025-2026,2026,2023
1,1 YOS,"$2,048,494","$2,150,917",NaN,NaN,NaN,2025-2026,2026,2023
2,2 YOS,"$2,296,274","$2,411,090","$2,525,901",NaN,NaN,2025-2026,2026,2023
3,3 YOS,"$2,378,870","$2,497,812","$2,616,754","$2,735,698",NaN,2025-2026,2026,2023
4,4 YOS,"$2,461,463","$2,584,539","$2,707,612","$2,830,685","$2,953,760",2025-2026,2026,2023
5,5 YOS,"$2,667,947","$2,801,346","$2,934,742","$3,068,140","$3,201,538",2025-2026,2026,2023
6,6 YOS,"$2,874,436","$3,018,158","$3,161,876","$3,305,598","$3,449,321",2025-2026,2026,2023
7,7 YOS,"$3,080,921","$3,234,968","$3,389,014","$3,543,059","$3,697,107",2025-2026,2026,2023
8,8 YOS,"$3,287,409","$3,451,779","$3,616,151","$3,780,524","$3,944,896",2025-2026,2026,2023
9,9 YOS,"$3,303,774","$3,468,962","$3,634,152","$3,799,340","$3,964,529",2025-2026,2026,2023


In [14]:
salary_min_18_clean1 = salary_min_18[
    ["Season", "Year", "CBA Regime", "YOS", "Year 1"]
].copy()

salary_min_18_clean1.head()

,Season,Year,CBA Regime,YOS,Year 1
0,2025-2026,2026,2023,0 YOS,"$1,272,870"
1,2025-2026,2026,2023,1 YOS,"$2,048,494"
2,2025-2026,2026,2023,2 YOS,"$2,296,274"
3,2025-2026,2026,2023,3 YOS,"$2,378,870"
4,2025-2026,2026,2023,4 YOS,"$2,461,463"


In [30]:
salary_min_18_clean1.columns = ["season", "year", "cba_id", "years_of_service", "minimum_salary"]

salary_min_18_clean1.head(25)

salary_min_18_clean2 = salary_min_18_clean1[salary_min_18_clean1["cba_id"].notna()]

salary_min_18_clean3 = salary_min_18_clean2[salary_min_18_clean2["season"] != "Season"]
salary_min_18_clean3.shape

(99, 5)

In [31]:
salary_min_18_clean3.notna().sum()

season              99
year                99
cba_id              99
years_of_service    99
minimum_salary      99
dtype: int64

In [32]:
# Remove dollar signs and commas
salary_min_18_clean3["minimum_salary"] = (
    salary_min_18_clean3["minimum_salary"]
    .str.replace(r"[$,]", "", regex=True)
    .astype(int)
)

# Remove the YOS suffix and surrounding whitespace
salary_min_18_clean3["years_of_service"] = (
    salary_min_18_clean3["years_of_service"]
    .str.replace("YOS", "", regex=False)
    .str.strip()
)

salary_min_18_clean3["year"] = salary_min_18_clean3["year"].astype(int)

salary_min_18_clean3.head()



,season,year,cba_id,years_of_service,minimum_salary
0,2025-2026,2026,2023,0,1272870
1,2025-2026,2026,2023,1,2048494
2,2025-2026,2026,2023,2,2296274
3,2025-2026,2026,2023,3,2378870
4,2025-2026,2026,2023,4,2461463


In [34]:
print(salary_min_18_clean3.shape)
salary_min_18_clean3.info()
print(salary_min_18_clean3.isna().sum())

(99, 5)
<class 'pandas.DataFrame'>
Index: 99 entries, 0 to 106
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   season            99 non-null     str  
 1   year              99 non-null     int64
 2   cba_id            99 non-null     str  
 3   years_of_service  99 non-null     str  
 4   minimum_salary    99 non-null     int64
dtypes: int64(2), str(3)
memory usage: 4.6 KB
season              0
year                0
cba_id              0
years_of_service    0
minimum_salary      0
dtype: int64


In [38]:
salary_min_18_clean3.duplicated(subset=["season", "years_of_service"]).sum()

np.int64(0)

In [40]:
salary_min_18_clean3.groupby(["season", "cba_id"]).size()

season     cba_id
2017-2018  2017      11
2018-2019  2017      11
2019-2020  2017      11
2020-2021  2017      11
2021-2022  2017      11
2022-2023  2017      11
2023-2024  2023      11
2024-2025  2023      11
2025-2026  2023      11
dtype: int64

In [42]:
salary_min_18_clean3["cba_id"] = ("cba_" + salary_min_18_clean3["cba_id"].astype(str))

salary_min_18_clean3.head(25)

,season,year,cba_id,years_of_service,minimum_salary
0,2025-2026,2026,cba_cba_2023,0,1272870
1,2025-2026,2026,cba_cba_2023,1,2048494
2,2025-2026,2026,cba_cba_2023,2,2296274
3,2025-2026,2026,cba_cba_2023,3,2378870
4,2025-2026,2026,cba_cba_2023,4,2461463
5,2025-2026,2026,cba_cba_2023,5,2667947
6,2025-2026,2026,cba_cba_2023,6,2874436
7,2025-2026,2026,cba_cba_2023,7,3080921
8,2025-2026,2026,cba_cba_2023,8,3287409
9,2025-2026,2026,cba_cba_2023,9,3303774


In [43]:
salary_min_18_clean3["cba_id"] = (
    salary_min_18_clean3["cba_id"]
    .str.replace(r"^(cba_)+", "cba_", regex=True)
)

In [44]:
salary_min_18_clean3["cba_id"].unique()

<StringArray>
['cba_2023', 'cba_2017']
Length: 2, dtype: str

In [45]:
# 1. Shape and data types
print(salary_min_18_clean3.shape)
salary_min_18_clean3.info()

# 2. Missing values
print(salary_min_18_clean3.isna().sum())

# 3. Duplicate primary keys
print(
    salary_min_18_clean3.duplicated(
        subset=["season", "years_of_service"]
    ).sum()
)

# 4. Observations per season
print(salary_min_18_clean3.groupby("season").size())

# 5. CBA identifiers
print(salary_min_18_clean3["cba_id"].unique())

(99, 5)
<class 'pandas.DataFrame'>
Index: 99 entries, 0 to 106
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   season            99 non-null     str  
 1   year              99 non-null     int64
 2   cba_id            99 non-null     str  
 3   years_of_service  99 non-null     str  
 4   minimum_salary    99 non-null     int64
dtypes: int64(2), str(3)
memory usage: 4.6 KB
season              0
year                0
cba_id              0
years_of_service    0
minimum_salary      0
dtype: int64
0
season
2017-2018    11
2018-2019    11
2019-2020    11
2020-2021    11
2021-2022    11
2022-2023    11
2023-2024    11
2024-2025    11
2025-2026    11
dtype: int64
<StringArray>
['cba_2023', 'cba_2017']
Length: 2, dtype: str


In [48]:
salary_min_18_final = salary_min_18_clean3.copy()

salary_min_18_final.to_csv(
    "../data/staging/contracts/s11b_salary_min_2018_2026.csv",
    index=False
)

In [50]:
salary_min_01.shape
salary_min_01.info()

<class 'pandas.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   YOS           35 non-null     str  
 1   2011-2012     35 non-null     str  
 2   2012-2013     35 non-null     str  
 3   2013-2014     35 non-null     str  
 4   2014-2015     35 non-null     str  
 5   2015-2016     35 non-null     str  
 6   2016-2017     35 non-null     str  
 7   CBA Regime    35 non-null     str  
 8   CBA Regime.1  12 non-null     str  
dtypes: str(9)
memory usage: 2.6 KB


In [51]:
salary_min_01

,YOS,2011-2012,2012-2013,2013-2014,2014-2015,2015-2016,2016-2017,CBA Regime,CBA Regime.1
0,0,"$473,604","$473,604","$490,180","$507,336","$525,093","$543,471",2011,NaN
1,1,"$762,195","$762,195","$788,872","$816,482","$845,059","$874,636",2011,NaN
2,2,"$854,389","$854,389","$884,293","$915,243","$947,276","$980,431",2011,NaN
3,3,"$885,120","$885,120","$916,099","$948,163","$981,348","$1,015,696",2011,NaN
4,4,"$915,852","$915,852","$947,907","$981,084","$1,015,421","$1,050,961",2011,NaN
5,5,"$992,680","$992,680","$1,027,424","$1,063,384","$1,100,602","$1,139,123",2011,NaN
6,6,"$1,069,509","$1,069,509","$1,106,942","$1,145,685","$1,185,784","$1,227,286",2011,NaN
7,7,"$1,146,337","$1,146,337","$1,186,459","$1,227,985","$1,270,964","$1,315,448",2011,NaN
8,8,"$1,223,166","$1,223,166","$1,265,977","$1,310,286","$1,356,146","$1,403,611",2011,NaN
9,9,"$1,229,255","$1,229,255","$1,272,279","$1,316,809","$1,362,897","$1,410,598",2011,NaN


In [52]:
salary_min_2011 = salary_min_01.iloc[0:11].copy()

salary_min_2005 = salary_min_01.iloc[12:23].copy()

salary_min_1998 = salary_min_01.iloc[24:35].copy()

print(salary_min_2011.shape)
print(salary_min_2005.shape)
print(salary_min_1998.shape)

(11, 9)
(11, 9)
(11, 9)


In [54]:
print(salary_min_2011.columns)
print(salary_min_2005.columns)
print(salary_min_1998.columns)

Index(['YOS', '2011-2012', '2012-2013', '2013-2014', '2014-2015', '2015-2016',
       '2016-2017', 'CBA Regime', 'CBA Regime.1'],
      dtype='str')
Index(['YOS', '2011-2012', '2012-2013', '2013-2014', '2014-2015', '2015-2016',
       '2016-2017', 'CBA Regime', 'CBA Regime.1'],
      dtype='str')
Index(['YOS', '2011-2012', '2012-2013', '2013-2014', '2014-2015', '2015-2016',
       '2016-2017', 'CBA Regime', 'CBA Regime.1'],
      dtype='str')


In [59]:
salary_min_1998

,YOS,2011-2012,2012-2013,2013-2014,2014-2015,2015-2016,2016-2017,CBA Regime,CBA Regime.1
24,0,"$287,500","$301,875","$316,969","$332,817","$349,458","$366,931","$385,277",1998
25,1,"$350,000","$385,000","$423,500","$465,850","$512,435","$563,679","$620,046",1998
26,2,"$425,000","$460,000","$498,500","$540,850","$587,435","$638,679","$695,046",1998
27,3,"$450,000","$485,000","$523,500","$565,850","$612,435","$663,679","$720,046",1998
28,4,"$475,000","$510,000","$548,500","$590,850","$637,435","$688,679","$745,046",1998
29,5,"$500,000","$572,500","$611,000","$653,350","$699,935","$751,179","$807,546",1998
30,6,"$500,000","$635,000","$673,500","$715,850","$762,435","$813,679","$870,046",1998
31,7,"$500,000","$697,500","$736,000","$778,350","$824,935","$876,179","$932,546",1998
32,8,"$500,000","$760,000","$798,500","$840,850","$887,435","$938,679","$995,046",1998
33,9,"$500,000","$885,000","$923,500","$965,850","$1,000,000","$1,000,000","$1,000,000",1998


In [60]:
salary_min_2011.columns = [
    "YOS",
    "2011-2012",
    "2012-2013",
    "2013-2014",
    "2014-2015",
    "2015-2016",
    "2016-2017",
    "CBA Regime",
    "CBA Regime.1"
]

salary_min_2005.columns = [
    "YOS",
    "2005-2006",
    "2006-2007",
    "2007-2008",
    "2008-2009",
    "2009-2010",
    "2010-2011",
    "CBA Regime",
    "CBA Regime.1"
]

salary_min_1998.columns = [
    "YOS",
    "1998-1999",
    "1999-2000",
    "2000-2001",
    "2001-2002",
    "2002-2003",
    "2003-2004",
    "2004-2005",
    "CBA Regime"
]

In [63]:
salary_min_2011

,YOS,2011-2012,2012-2013,2013-2014,2014-2015,2015-2016,2016-2017,CBA Regime,CBA Regime.1
0,0,"$473,604","$473,604","$490,180","$507,336","$525,093","$543,471",2011,NaN
1,1,"$762,195","$762,195","$788,872","$816,482","$845,059","$874,636",2011,NaN
2,2,"$854,389","$854,389","$884,293","$915,243","$947,276","$980,431",2011,NaN
3,3,"$885,120","$885,120","$916,099","$948,163","$981,348","$1,015,696",2011,NaN
4,4,"$915,852","$915,852","$947,907","$981,084","$1,015,421","$1,050,961",2011,NaN
5,5,"$992,680","$992,680","$1,027,424","$1,063,384","$1,100,602","$1,139,123",2011,NaN
6,6,"$1,069,509","$1,069,509","$1,106,942","$1,145,685","$1,185,784","$1,227,286",2011,NaN
7,7,"$1,146,337","$1,146,337","$1,186,459","$1,227,985","$1,270,964","$1,315,448",2011,NaN
8,8,"$1,223,166","$1,223,166","$1,265,977","$1,310,286","$1,356,146","$1,403,611",2011,NaN
9,9,"$1,229,255","$1,229,255","$1,272,279","$1,316,809","$1,362,897","$1,410,598",2011,NaN


In [87]:
salary_min_2011_long = pd.melt(salary_min_2011,
id_vars = ["YOS", "CBA Regime"],
value_vars = ["2011-2012",
        "2012-2013",
        "2013-2014",
        "2014-2015",
        "2015-2016",
        "2016-2017"],
var_name = "season",
value_name ="minimum_salary"
)

salary_min_2011_long.head()

,YOS,CBA Regime,season,minimum_salary
0,0,2011,2011-2012,"$473,604"
1,1,2011,2011-2012,"$762,195"
2,2,2011,2011-2012,"$854,389"
3,3,2011,2011-2012,"$885,120"
4,4,2011,2011-2012,"$915,852"


In [88]:
salary_min_2005_long = pd.melt(salary_min_2005,
id_vars = ["YOS", "CBA Regime"],
value_vars = ["2005-2006",
"2006-2007",
"2007-2008",
"2008-2009",
"2009-2010",
"2010-2011"],
var_name = "season",
value_name = "minimum_salary"
)

salary_min_2005_long.head()


,YOS,CBA Regime,season,minimum_salary
0,0,2005,2005-2006,"$398,762"
1,1,2005,2005-2006,"$641,748"
2,2,2005,2005-2006,"$719,373"
3,3,2005,2005-2006,"$745,248"
4,4,2005,2005-2006,"$771,123"


In [89]:
salary_min_1998_long = pd.melt(salary_min_1998,
id_vars = ["YOS", "CBA Regime"],
value_vars = ["1998-1999",
"1999-2000",
"2000-2001",
"2001-2002",
"2002-2003",
"2003-2004",
"2004-2005"],
var_name = "season",
value_name = "minimum_salary"
)

salary_min_1998_long.head()

,YOS,CBA Regime,season,minimum_salary
0,0,1998,1998-1999,"$287,500"
1,1,1998,1998-1999,"$350,000"
2,2,1998,1998-1999,"$425,000"
3,3,1998,1998-1999,"$450,000"
4,4,1998,1998-1999,"$475,000"


In [90]:
salary_min_01_17 = pd.concat([salary_min_2011_long, salary_min_2005_long, salary_min_1998_long], ignore_index=True )

salary_min_01_17.head()

,YOS,CBA Regime,season,minimum_salary
0,0,2011,2011-2012,"$473,604"
1,1,2011,2011-2012,"$762,195"
2,2,2011,2011-2012,"$854,389"
3,3,2011,2011-2012,"$885,120"
4,4,2011,2011-2012,"$915,852"


In [91]:
salary_min_01_17.info()

<class 'pandas.DataFrame'>
RangeIndex: 209 entries, 0 to 208
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   YOS             209 non-null    str  
 1   CBA Regime      209 non-null    str  
 2   season          209 non-null    str  
 3   minimum_salary  209 non-null    str  
dtypes: str(4)
memory usage: 6.7 KB


In [92]:
salary_min_18_final.info()

<class 'pandas.DataFrame'>
Index: 99 entries, 0 to 106
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   season            99 non-null     str  
 1   year              99 non-null     int64
 2   cba_id            99 non-null     str  
 3   years_of_service  99 non-null     str  
 4   minimum_salary    99 non-null     int64
dtypes: int64(2), str(3)
memory usage: 4.6 KB


In [93]:
salary_min_01_17["year"] = (salary_min_01_17["season"].str[-4:].astype(int))

salary_min_01_17.head()

,YOS,CBA Regime,season,minimum_salary,year
0,0,2011,2011-2012,"$473,604",2012
1,1,2011,2011-2012,"$762,195",2012
2,2,2011,2011-2012,"$854,389",2012
3,3,2011,2011-2012,"$885,120",2012
4,4,2011,2011-2012,"$915,852",2012


In [94]:
salary_min_01_combined = salary_min_01_17[["season", "year", "CBA Regime", "YOS", "minimum_salary"]].copy()

salary_min_01_combined.head()

,season,year,CBA Regime,YOS,minimum_salary
0,2011-2012,2012,2011,0,"$473,604"
1,2011-2012,2012,2011,1,"$762,195"
2,2011-2012,2012,2011,2,"$854,389"
3,2011-2012,2012,2011,3,"$885,120"
4,2011-2012,2012,2011,4,"$915,852"


In [95]:
renamed_columns = ["season", "year", "cba_id", "years_of_service", "minimum_salary"]

salary_min_01_combined.columns = renamed_columns

salary_min_01_combined.head()

,season,year,cba_id,years_of_service,minimum_salary
0,2011-2012,2012,2011,0,"$473,604"
1,2011-2012,2012,2011,1,"$762,195"
2,2011-2012,2012,2011,2,"$854,389"
3,2011-2012,2012,2011,3,"$885,120"
4,2011-2012,2012,2011,4,"$915,852"


In [96]:
salary_min_18_final.head()

,season,year,cba_id,years_of_service,minimum_salary
0,2025-2026,2026,cba_2023,0,1272870
1,2025-2026,2026,cba_2023,1,2048494
2,2025-2026,2026,cba_2023,2,2296274
3,2025-2026,2026,cba_2023,3,2378870
4,2025-2026,2026,cba_2023,4,2461463


In [97]:
# 1. Standardize CBA identifiers
salary_min_01_combined["cba_id"] = (
    "cba_" + salary_min_01_combined["cba_id"]
    .astype(str)
)

# 2. Clean salary values and convert to integers
salary_min_01_combined["minimum_salary"] = (
    salary_min_01_combined["minimum_salary"]
    .str.replace(r"[$,]", "", regex=True)
    .astype(int)
)

# 3. Verify
print(salary_min_01_combined["cba_id"].unique())
print(salary_min_01_combined.dtypes)
salary_min_01_combined.head()

<StringArray>
['cba_2011', 'cba_2005', 'cba_1998']
Length: 3, dtype: str
season                str
year                int64
cba_id                str
years_of_service      str
minimum_salary      int64
dtype: object


,season,year,cba_id,years_of_service,minimum_salary
0,2011-2012,2012,cba_2011,0,473604
1,2011-2012,2012,cba_2011,1,762195
2,2011-2012,2012,cba_2011,2,854389
3,2011-2012,2012,cba_2011,3,885120
4,2011-2012,2012,cba_2011,4,915852


In [98]:
salary_min_01_combined.info()

<class 'pandas.DataFrame'>
RangeIndex: 209 entries, 0 to 208
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   season            209 non-null    str  
 1   year              209 non-null    int64
 2   cba_id            209 non-null    str  
 3   years_of_service  209 non-null    str  
 4   minimum_salary    209 non-null    int64
dtypes: int64(2), str(3)
memory usage: 8.3 KB


In [100]:
salary_min_01_combined["cba_id"].nunique()

3

In [101]:
salary_min_01_combined["cba_id"] = (
    salary_min_01_combined["cba_id"]
    .replace({"cba_1998": "cba_1999"})
)

In [102]:
salary_min_01_combined.duplicated(
    subset=["season", "years_of_service"]
).sum()

np.int64(0)

In [103]:
salary_min_01_combined.groupby("season").size()

season
1998-1999    11
1999-2000    11
2000-2001    11
2001-2002    11
2002-2003    11
2003-2004    11
2004-2005    11
2005-2006    11
2006-2007    11
2007-2008    11
2008-2009    11
2009-2010    11
2010-2011    11
2011-2012    11
2012-2013    11
2013-2014    11
2014-2015    11
2015-2016    11
2016-2017    11
dtype: int64

In [104]:
print((salary_min_01_combined["minimum_salary"] > 0).all())

print(
    (
        salary_min_01_combined["year"]
        == salary_min_01_combined["season"].str[-4:].astype(int)
    ).all()
)

True
True


In [105]:
salary_min_01_final = salary_min_01_combined[
    salary_min_01_combined["year"] >= 2001
].copy()

print(salary_min_01_final.shape)

(187, 5)


In [108]:
salary_min_01_final.head(15)

,season,year,cba_id,years_of_service,minimum_salary
0,2011-2012,2012,cba_2011,0,473604
1,2011-2012,2012,cba_2011,1,762195
2,2011-2012,2012,cba_2011,2,854389
3,2011-2012,2012,cba_2011,3,885120
4,2011-2012,2012,cba_2011,4,915852
5,2011-2012,2012,cba_2011,5,992680
6,2011-2012,2012,cba_2011,6,1069509
7,2011-2012,2012,cba_2011,7,1146337
8,2011-2012,2012,cba_2011,8,1223166
9,2011-2012,2012,cba_2011,9,1229255


In [109]:
salary_min_18_final["years_of_service"] = (
    salary_min_18_final["years_of_service"]
    .replace({"10": "10+"})
)

In [116]:
salary_min_01_final.head(15)

,season,year,cba_id,years_of_service,minimum_salary
0,2011-2012,2012,cba_2011,0,473604
1,2011-2012,2012,cba_2011,1,762195
2,2011-2012,2012,cba_2011,2,854389
3,2011-2012,2012,cba_2011,3,885120
4,2011-2012,2012,cba_2011,4,915852
5,2011-2012,2012,cba_2011,5,992680
6,2011-2012,2012,cba_2011,6,1069509
7,2011-2012,2012,cba_2011,7,1146337
8,2011-2012,2012,cba_2011,8,1223166
9,2011-2012,2012,cba_2011,9,1229255


In [117]:
salary_min_final = pd.concat(
    [salary_min_01_final, salary_min_18_final],
    ignore_index=True
)

print(salary_min_final.shape)

print(
    salary_min_final.duplicated(
        subset=["season", "years_of_service"]
    ).sum()
)

print(salary_min_final.groupby("season").size())

(286, 5)
0
season
2000-2001    11
2001-2002    11
2002-2003    11
2003-2004    11
2004-2005    11
2005-2006    11
2006-2007    11
2007-2008    11
2008-2009    11
2009-2010    11
2010-2011    11
2011-2012    11
2012-2013    11
2013-2014    11
2014-2015    11
2015-2016    11
2016-2017    11
2017-2018    11
2018-2019    11
2019-2020    11
2020-2021    11
2021-2022    11
2022-2023    11
2023-2024    11
2024-2025    11
2025-2026    11
dtype: int64


In [118]:
salary_min_01_final.to_csv(
    "../data/staging/contracts/s11b_salary_min_2001_2017.csv",
    index=False
)

In [119]:
salary_min_final.to_csv(
    "../data/staging/contracts/s11b_salary_min_final.csv",
    index=False
)

In [121]:
salary_min_check = pd.read_csv("../data/staging/contracts/s11b_salary_min_final.csv")

salary_min_check.shape

(286, 5)